In [1]:
!pip install seaborn

In [2]:
!pip install scipy

Define Metrics
We quantify "risk" using three key metrics:

Claim Frequency: Number of policies with claims > 0 / Total number of policies
Claim Severity: Average TotalClaims among records where TotalClaims > 0
Margin: TotalPremium - TotalClaims

In [5]:
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv('../MachineLearningRating_v3.txt', delimiter='|', encoding='utf-8', low_memory=False) 

# Basic metrics
df['HasClaim'] = df['TotalClaims'] > 0
df['Margin'] = df['TotalPremium'] - df['TotalClaims']
claim_freq = df['HasClaim'].mean()
claim_sev = df[df['HasClaim']]['TotalClaims'].mean()
margin_avg = df['Margin'].mean()
   
print(f"Claim Frequency: {claim_freq:.2%}")
print(f"Claim Severity (avg claims given claim): {claim_sev:.2f}")
print(f"Average Margin: {margin_avg:.2f}") 

Claim Frequency: 0.28%
Claim Severity (avg claims given claim): 23273.39
Average Margin: -2.96


Hypothesis Testing
We test 3 null hypotheses using appropriate statistical methods.
H₀ (1): No risk difference across Provinces
Use ANOVA to test if Claim Severity differs across provinces.
If p < 0.05 => Reject H₀ => Some provinces have different risk levels

In [ ]:
province_groups = df[df['HasClaim']].groupby('Province')['TotalClaims'].apply(list)
province_anova = stats.f_oneway(*province_groups)
print("ANOVA Province vs Claim Severity:")
print(f"F-statistic: {province_anova.statistic:.4f}, p-value: {province_anova.pvalue:.4f}")

Interpretation for Ho(1)
Sice p-value = 0.0000 < 0.05, therefore we reject the null hypothesis.
This indicates that there is statistically high difference across province which means the province has statistically significant risk

H₀ (2): No risk difference between ZipCode( in our case Postal Codes)
Use PostalCode to test claim frequency differences.

In [ ]:
postal_groups = df.groupby('PostalCode')['HasClaim'].apply(list)
postal_anova = stats.f_oneway(*postal_groups)
print("ANOVA: PostalCode vs Claim Frequency")
print(f"F-statistic: {postal_anova.statistic:.4f}, p-value: {postal_anova.pvalue:.4f}")

Interpretation for Ho(2)
Our p-value = 0.0000 < 0.05, therefore we reject the null hypothesis.
This means that risk (claim frequency or severity) differs significantly between postal code regions.

Ho (3) No significant margin (profit) difference between Cover Groups
We test if Margin differs significantly across CoverGroup using ANOVA.
If p < 0.05 → Reject H₀ → Some CoverGroups are more/less profitable|\

In [ ]:
groups = df.groupby('CoverGroup')['Margin'].apply(list)
margin_anova = stats.f_oneway(*groups)
print("ANOVA CoverGroup vs Margin:")
print(f"F-statistic: {margin_anova.statistic:.4f}, p-value: {margin_anova.pvalue:.4f}")

Interpretation for Ho(3)
Since p-value = 0.0000 < 0.05, we reject the null hypothesis.
This indicates that at least one CoverGroup has a statistically different average profit margin.
ACIS should review its underwriting and pricing strategies for these groups to ensure profitability.

Ho (4) No risk difference between Men and Women
Use t-test to compare Claim Severity between genders.

In [ ]:
men = df[(df['Title'] == 'Mr') & (df['HasClaim'])]['TotalClaims']
women = df[(df['Title'] != 'Mr') & (df['HasClaim'])]['TotalClaims']
ttest_gender = stats.ttest_ind(men, women, equal_var=False)
print("T-test: Gender vs Claim Severity")
print(f"t-statistic: {ttest_gender.statistic:.4f}, p-value: {ttest_gender.pvalue:.4f}")

Interpretaion for Ho (4) 
Since our p-value is >0.5 we fail to reject the null hypothesis. This means There's no statistically significant difference in claim severity between genders in your dataset.
This implies that gender is not a key risk driver for claim severity, and thus, should not influence premium pricing.

In [ ]:
#Visualization Examples
# Boxplot for Margin by CoverGroup
plt.figure(figsize=(10,5))
sns.boxplot(x='CoverGroup', y='Margin', data=df)
plt.title("Margin by Cover Group")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# KDE for Claim Severity by Gender
plt.figure(figsize=(8,5))
sns.kdeplot(men, label='Men', shade=True)
sns.kdeplot(women, label='Women', shade=True)
plt.legend()
plt.title("Claim Severity by Gender")
plt.show()